# OneLake → Azure AI Search with Content Understanding skill

Builds an end-to-end pipeline:

1. Reads files from a **Microsoft Fabric Lakehouse** (OneLake `Files/`).
2. Runs the **Content Understanding skill** to extract structured content **with page numbers**.
3. Splits per page → embeds with **Azure OpenAI** → writes to an **Azure AI Search** vector index with `page_number` carried onto every chunk.

## Prerequisites

- Azure AI Search service (Basic or higher) **in the same tenant** as your Fabric workspace, with **RBAC enabled** (Settings → Keys → *Role-based access control*).
- Your identity (and the deployed managed identity) has **Search Service Contributor** + **Search Index Data Contributor** on the search service.
- Search service has a **system-assigned managed identity** with **Contributor** on the Fabric workspace.
- Fabric admin: *Allow apps running outside of Fabric to access data via OneLake* is **enabled**.
- An **Azure AI Foundry resource** (provisioned in [ai.azure.com](https://ai.azure.com); ARM type `Microsoft.CognitiveServices/accounts`, kind `AIServices`) — this is what the Content Understanding skill bills against. It must be in a region that supports Content Understanding for the modality you use:
  - Americas: `westus`, `westus3`, `eastus`, `eastus2`, `southcentralus`
  - Europe: `northeurope`, `westeurope`, `swedencentral`, `uksouth`
  - Asia Pacific: `australiaeast`, `japaneast`, `southeastasia`
  - Authoritative matrix (per modality + API version): https://learn.microsoft.com/azure/ai-services/content-understanding/language-region-support
  - Audio / video / some prebuilt analyzers are in **fewer** regions than document — check the matrix if you plan to extend beyond documents.
  - Bind to it via **managed identity** (this notebook: the search service MI needs the **Cognitive Services User** role on the Foundry resource).
- The search service MI also needs **Cognitive Services OpenAI User** on the Azure OpenAI resource (used by the embedding skill + vectorizer).
- An **Azure OpenAI** resource with a `text-embedding-3-large` deployment.
- Files already uploaded to `Files/engagements/project001/raw_files` in the lakehouse (PDF / DOCX / PPTX work best for page numbers).

## Environment setup (run once, in a terminal)

Create a local virtual environment and register it as a Jupyter kernel so this notebook can use it.

**Windows / PowerShell**

```powershell
cd notebooks
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name fti-onelake --display-name "Python (fti-onelake)"
Copy-Item .env.example .env   # then edit .env with your values
```

**macOS / Linux**

```bash
cd notebooks
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name fti-onelake --display-name "Python (fti-onelake)"
cp .env.example .env          # then edit .env with your values
```

Then in this notebook, select the **Python (fti-onelake)** kernel (top-right kernel picker in VS Code, or *Kernel → Change kernel* in Jupyter Lab).

The `%pip install` cell below is a no-op once the venv is active — leave it in for Colab / Codespaces users, or skip it.

## 1. Install + import

In [ ]:
# Skip if you already installed via `pip install -r requirements.txt` in your venv.
%pip install --quiet -r requirements.txt

In [8]:
import os, json, time, requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
load_dotenv(override=True)

SEARCH_ENDPOINT  = os.environ['SEARCH_ENDPOINT']
API_VERSION      = '2025-11-01-Preview'                    # supports ContentUnderstandingSkill (or use 2026-04-01 GA)

FABRIC_WORKSPACE_GUID = os.environ['FABRIC_WORKSPACE_GUID']
LAKEHOUSE_GUID        = os.environ['LAKEHOUSE_GUID']
LAKEHOUSE_SUBPATH     = 'engagements/project001/raw_files'

# ---- Azure AI Foundry resource (billing target for Content Understanding) ----
# This is the Foundry resource you create in https://ai.azure.com
# (ARM type: Microsoft.CognitiveServices/accounts, kind: AIServices).
# Endpoint is on the resource's 'Keys and Endpoint' blade.
FOUNDRY_ENDPOINT = os.environ['FOUNDRY_ENDPOINT']           # https://<name>.cognitiveservices.azure.com

# The Content Understanding skill's 'subdomainUrl' must be the BASE account subdomain
# only — no path/query (a project endpoint like '.../api/projects/<proj>' is rejected
# with: "'SubdomainUrl' parameter should reference the base subdomain url...").
from urllib.parse import urlsplit
_fp = urlsplit(FOUNDRY_ENDPOINT)
FOUNDRY_SUBDOMAIN = f'{_fp.scheme}://{_fp.netloc}'

# ---- Azure OpenAI for embeddings ----
# AOAI_ENDPOINT should be the raw AOAI resource for managed-identity auth:
#   https://<aoai-name>.openai.azure.com
# (An APIM gateway also works only if its policy validates the caller's Entra ID
#  token and forwards to /openai/deployments/{deployment}/embeddings.)
AOAI_ENDPOINT   = os.environ['AOAI_ENDPOINT']
AOAI_EMBED_DEPL = os.environ.get('AOAI_EMBED_DEPL', 'text-embedding-3-small')
AOAI_EMBED_MODEL= os.environ.get('AOAI_EMBED_MODEL', 'text-embedding-3-small')
EMBED_DIMS      = int(os.environ.get('EMBED_DIMS', '1536'))  # small=1536, large=3072

# Search builds '{resourceUri}/openai/deployments/{deployment}/embeddings' itself, so
# resourceUri must be the bare AOAI origin — any extra path (e.g. '/openai/v1') yields a 404.
AOAI_RESOURCE_URI = f'{urlsplit(AOAI_ENDPOINT).scheme}://{urlsplit(AOAI_ENDPOINT).netloc}'

PREFIX    = 'fti-onelake-cu'
DS_NAME   = f'{PREFIX}-ds'
SS_NAME   = f'{PREFIX}-ss'
IDX_NAME  = f'{PREFIX}-idx'
IDXR_NAME = f'{PREFIX}-idxr'

# ---- Entra ID auth (managed identity in Azure; az login / VS Code identity locally) ----
# Your identity needs these RBAC roles on the Azure AI Search service:
#   - 'Search Service Contributor'    (create/update indexers, indexes, skillsets, data sources)
#   - 'Search Index Data Contributor' (write documents) or 'Search Index Data Reader' (query only)
# The search service must also have RBAC enabled (authOptions → 'Role-based access control').
#
# IMPORTANT: DefaultAzureCredential walks a chain (env → managed identity → VS Code →
# shared cache → Azure CLI → Azure PowerShell) and may pick a DIFFERENT account/tenant
# than your `az login`. If the token is issued for the wrong identity/tenant you'll get a
# 403 even though the role is assigned. Pin AZURE_TENANT_ID in .env to the tenant that owns
# the search service so every credential in the chain targets it.
TENANT_ID = os.environ.get('AZURE_TENANT_ID')  # e.g. 7b3b2559-0c78-44e7-8c57-8455603aca36
_cred_kwargs = {}
if TENANT_ID:
    _cred_kwargs = {
        'shared_cache_tenant_id':        TENANT_ID,
        'visual_studio_code_tenant_id':  TENANT_ID,
        'interactive_browser_tenant_id': TENANT_ID,
    }
credential   = DefaultAzureCredential(**_cred_kwargs)
SEARCH_SCOPE = 'https://search.azure.com/.default'

# Diagnostic: show WHICH principal/tenant the token belongs to (this is what Search authorizes).
def whoami():
    import base64
    tok = credential.get_token(SEARCH_SCOPE).token
    p = tok.split('.')[1]
    claims = json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))
    print('token identity → oid:', claims.get('oid'),
          '| upn/appid:', claims.get('upn') or claims.get('unique_name') or claims.get('appid'),
          '| tenant:', claims.get('tid'))
    return claims

def _headers():
    token = credential.get_token(SEARCH_SCOPE).token
    return {'Content-Type': 'application/json', 'Authorization': f'Bearer {token}'}

def put(path, body):
    r = requests.put(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=_headers(), data=json.dumps(body))
    if r.status_code >= 300: raise RuntimeError(f'{r.status_code}: {r.text}')
    return r
def post(path):
    r = requests.post(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=_headers())
    if r.status_code >= 300: raise RuntimeError(f'{r.status_code}: {r.text}')
    return r
def get(path):
    return requests.get(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=_headers()).json()

print('Config loaded.')
whoami()  # confirm this oid/tenant matches the identity you granted the Search role to

Config loaded.


## 2. OneLake data source

**Note:** the OneLake indexer does *not* accept the `https://onelake.dfs.fabric.microsoft.com/...` URL. It needs:

- `connectionString = ResourceId=<workspace GUID>`
- `container.name   = <lakehouse GUID>`
- `container.query  = <subfolder under Files/>`  *(no leading `Files/`)*

In [9]:
data_source = {
    'name': DS_NAME,
    'type': 'onelake',
    'credentials': { 'connectionString': f'ResourceId={FABRIC_WORKSPACE_GUID}' },
    'container':   { 'name': LAKEHOUSE_GUID, 'query': LAKEHOUSE_SUBPATH },
}
put(f"/datasources('{DS_NAME}')", data_source)
print('Data source created:', DS_NAME)

Data source created: fti-onelake-cu-ds


## 3. Index — one document per chunk

Each chunk from Content Understanding becomes one search doc. `page_number` = `pageNumberFrom`, `page_to` = `pageNumberTo` (a chunk can span pages).

In [10]:
index_def = {
    'name': IDX_NAME,
    'fields': [
        {'name': 'chunk_id',     'type': 'Edm.String', 'key': True, 'filterable': True, 'sortable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id',    'type': 'Edm.String', 'filterable': True},
        {'name': 'file_name',    'type': 'Edm.String', 'filterable': True, 'searchable': True},
        {'name': 'file_path',    'type': 'Edm.String', 'filterable': True},
        {'name': 'page_number',  'type': 'Edm.Int32',  'filterable': True, 'sortable': True, 'facetable': True},
        {'name': 'page_to',      'type': 'Edm.Int32',  'filterable': True, 'sortable': True},
        {'name': 'content',      'type': 'Edm.String', 'searchable': True},
        {'name': 'content_vector','type': 'Collection(Edm.Single)', 'searchable': True,
         'dimensions': EMBED_DIMS, 'vectorSearchProfile': 'hnsw-aoai'},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'hnsw-algo', 'kind': 'hnsw'}],
        'profiles':   [{'name': 'hnsw-aoai', 'algorithm': 'hnsw-algo', 'vectorizer': 'aoai-vec'}],
        'vectorizers':[{'name': 'aoai-vec', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri':  AOAI_RESOURCE_URI,
                'deploymentId': AOAI_EMBED_DEPL,
                'modelName':    AOAI_EMBED_MODEL,
            }
        }]
    },
    'semantic': {
        'configurations': [{
            'name': 'sem-default',
            'prioritizedFields': {
                'titleField': {'fieldName': 'file_name'},
                'prioritizedContentFields': [{'fieldName': 'content'}]
            }
        }]
    }
}
put(f"/indexes('{IDX_NAME}')", index_def)
print('Index created:', IDX_NAME)

Index created: fti-onelake-cu-idx


## 4. Skillset — Content Understanding + AOAI embedding

The Content Understanding skill **chunks the document itself** (no Text Split skill needed). Each chunk in `text_sections` carries `locationMetadata.pageNumberFrom` / `pageNumberTo` — but **only if you set `extractionOptions: ["locationMetadata"]`**. The index projection writes one search doc per chunk.

In [11]:
skillset = {
    'name': SS_NAME,
    'description': 'Content Understanding extraction + chunking → AOAI embeddings',
    'cognitiveServices': {
        '@odata.type': '#Microsoft.Azure.Search.AIServicesByIdentity',
        'description': 'Azure AI Foundry resource (billing for Content Understanding skill)',
        'subdomainUrl': FOUNDRY_SUBDOMAIN,
        'identity': None  # null = system-assigned MI of the search service
    },
    # The search service MI needs the 'Cognitive Services User' role on the Foundry resource.
    # --- alt: bind by key instead (quick start / dev) ---
    # 'cognitiveServices': {
    #     '@odata.type': '#Microsoft.Azure.Search.AIServicesByKey',
    #     'description': 'Azure AI Foundry resource',
    #     'key': os.environ['FOUNDRY_KEY'],
    #     'subdomainUrl': FOUNDRY_SUBDOMAIN,
    # },
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
            'name': 'cu',
            'description': 'Extract text + page metadata, chunk semantically',
            'context': '/document',
            'extractionOptions': ['locationMetadata'],
            'chunkingProperties': {
                'unit': 'characters',
                'maximumLength': 2000,
                'overlapLength': 200
            },
            'inputs':  [{'name': 'file_data', 'source': '/document/file_data'}],
            'outputs': [
                {'name': 'text_sections',     'targetName': 'text_sections'},
                {'name': 'normalized_images', 'targetName': 'normalized_images'}
            ]
        },
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed',
            'context': '/document/text_sections/*',
            'resourceUri':  AOAI_RESOURCE_URI,
            'deploymentId': AOAI_EMBED_DEPL,
            'modelName':    AOAI_EMBED_MODEL,
            'dimensions':   EMBED_DIMS,
            'inputs':  [{'name': 'text',      'source': '/document/text_sections/*/content'}],
            'outputs': [{'name': 'embedding', 'targetName': 'content_vector'}]
        }
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': IDX_NAME,
            'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/text_sections/*',
            'mappings': [
                {'name': 'file_name',     'source': '/document/metadata_storage_name'},
                {'name': 'file_path',     'source': '/document/metadata_storage_path'},
                {'name': 'page_number',   'source': '/document/text_sections/*/locationMetadata/pageNumberFrom'},
                {'name': 'page_to',       'source': '/document/text_sections/*/locationMetadata/pageNumberTo'},
                {'name': 'content',       'source': '/document/text_sections/*/content'},
                {'name': 'content_vector','source': '/document/text_sections/*/content_vector'}
            ]
        }],
        'parameters': { 'projectionMode': 'skipIndexingParentDocuments' }
    }
}
put(f"/skillsets('{SS_NAME}')", skillset)
print('Skillset created:', SS_NAME)

Skillset created: fti-onelake-cu-ss


## 5. Indexer

`allowSkillsetToReadFileData = true` is what exposes `/document/file_data` to the Content Understanding skill.

In [12]:
indexer = {
    'name': IDXR_NAME,
    'dataSourceName':  DS_NAME,
    'targetIndexName': IDX_NAME,
    'skillsetName':    SS_NAME,
    'parameters': {
        'batchSize': 1,
        'maxFailedItems': 5,
        'configuration': {
            'dataToExtract': 'contentAndMetadata',
            'parsingMode': 'default',
            'allowSkillsetToReadFileData': True,
            'indexedFileNameExtensions': '.pdf,.docx,.pptx,.xlsx,.html,.txt,.md'
        }
    },
    'fieldMappings': [
        {'sourceFieldName': 'metadata_storage_path', 'targetFieldName': 'parent_id', 'mappingFunction': {'name': 'base64Encode'}}
    ]
}
put(f"/indexers('{IDXR_NAME}')", indexer)
post(f"/indexers('{IDXR_NAME}')/search.run")
print('Indexer started:', IDXR_NAME)

Indexer started: fti-onelake-cu-idxr


## 6. Watch status

In [13]:
for _ in range(60):
    status = get(f"/indexers('{IDXR_NAME}')/search.status")
    last = status.get('lastResult') or {}
    print(status.get('status'), '|', last.get('status'),
          '| processed:', last.get('itemsProcessed'),
          '| failed:', last.get('itemsFailed'),
          '| errors:', (last.get('errors') or [])[:2])
    if last.get('status') in ('success', 'transientFailure', 'persistentFailure'):
        break
    time.sleep(15)

running | None | processed: None | failed: None | errors: []
running | inProgress | processed: 0 | failed: 0 | errors: []
running | success | processed: 0 | failed: 0 | errors: []


## 7. Query — verify page numbers come back

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

client = SearchClient(SEARCH_ENDPOINT, IDX_NAME, credential)
results = client.search(
    search_text='termination clause',
    vector_queries=[VectorizableTextQuery(text='termination clause', k_nearest_neighbors=5, fields='content_vector')],
    select=['file_name', 'page_number', 'content'],
    top=5,
)
for r in results:
    print(f"{r['file_name']} p.{r['page_number']}: {r['content'][:160]}…")

## Cleanup (optional)

In [ ]:
for path in [f"/indexers('{IDXR_NAME}')", f"/skillsets('{SS_NAME}')", f"/indexes('{IDX_NAME}')", f"/datasources('{DS_NAME}')"]:
    r = requests.delete(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=_headers())
    print(path, '→', r.status_code)